In [ ]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json

In [ ]:
load_dotenv(override=True)

In [ ]:
openai = OpenAI()
model_name = "gpt-5.2"

In [ ]:
def show(text):
    Console(force_jupyter=True).print(text)

In [ ]:
todos, completed = [], []

In [ ]:
def get_todos_report():
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index+1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index+1}: [red]{todo}[/red]\n"
    Console(force_jupyter=True).print(result)
    return result

def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todos_report()

def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console(force_jupyter=True).print(completion_notes)
    return get_todos_report()

In [ ]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
            },
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
complete_todo_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "type": "integer",
                "description": "The 1-based index of the todo to mark as complete"
            },
            "completion_notes": {
                "type": "string",
                "description": "Notes about how you completed the todo in rich console markup"
            }
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": complete_todo_json}
]

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        if not tool:
            raise ValueError(f"Tool '{tool_name}' not found — check that the JSON schema name matches the Python function name")
        result = tool(**arguments)
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(
            model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none"
        )
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [ ]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

user_message = """
Estimate the total cost of a road trip from San Francisco to New York.
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

In [ ]:
todos, completed = [], []
loop(messages)